# Paper experiment reproduction

In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys
from pathlib import Path
import numpy as np
import xgboost as xgb
import talib
from sklearn.linear_model import LinearRegression
import re
import ast
import warnings
import time
import functools
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "trading_graph.py").exists())
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "reproduction" / "eval"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")
from default_config import DEFAULT_CONFIG
from trading_graph import TradingGraph
import static_util

# Gemini repeats this on every request (about 5 per segment) and it would flood the cell output
warnings.filterwarnings("ignore", message=".*uses fixed sampling defaults", category=UserWarning)

In Appendix E, section 2 "Data Collection and Asset Selection" there is stated the following:

> For each asset, we collect 5,000 historical 1-hour and 4-hour bars. To ensure fairness and consistency across assets, we apply the same fixed bar count to all instruments—including those with limited trading hours, such as QQQ. As a result, assets with lower intraday trading frequency span a longer historical period (up to ten years), reflecting the reduced density of available candlestick intervals. From this data, we randomly sample 100 evaluation segments per asset. Each segment consists of 100 consecutive candlesticks, with the final 3 candlesticks removed during inference to ensure the system does not observe the true market outcome during prediction. The final three candlesticks are reserved for validating the correctness of the LLM’s prediction.

Meanwhile, in the [run_analysis()](../../web_interface.py#L301) function in the provided [web_interface.py](../../web_interface.py) file they use just 45 last candlesticks from fetched data, hence I use my own code for cleaner paper reproduction.

In [2]:
SEED = 13

## Evaluation protocol implementation

#### Directional accuracy

From paper:

>Accuracy is defined as $\alpha = \frac{\mathbb{C}}{\mathcal{T}}$, where $\mathbb{C}$ is the number of correct predictions and $\mathcal{T}$ is the total evaluated. Each test yields a score from $0$ to $3$, and the average is computed over all samples.

In [3]:
def directional_accuracy(df, decision):
  reference = float(df.iloc[-4]["Close"])

  future_closes = (
    df.iloc[-3:]["Close"].astype(float)
  )

  if decision == "LONG":
    correct = (future_closes > reference).sum()
  elif decision == "SHORT":
    correct = (future_closes < reference).sum()
  else:
    raise ValueError(f"Unknown decision: {decision}")

  return correct / 3

#### Rate of Return (RoR) estimators

The following formulas are obtained from an older version of a paper [arXiv:2509.09995v2 [cs.CE] 15 Sep 2025](https://arxiv.org/abs/2509.09995v2):


$$
\mathcal{R}_{cc} =
\begin{cases}
\frac{\mathrm{Close}_{t+3}}{\mathrm{Close}_t} - 1 & \text{if LONG} \\[6pt]
1 - \frac{\mathrm{Close}_{t+3}}{\mathrm{Close}_t} & \text{if SHORT}
\end{cases}
$$


In [4]:
def ror_cc(df, decision):
  current_close = float(df.iloc[-4]["Close"])

  future_close = float(df.iloc[-1]["Close"])

  if decision == "LONG":
    result = future_close / current_close - 1
  elif decision == "SHORT":
    result = 1 - future_close / current_close
  else:
    raise ValueError(f"Unknown decision: {decision}")

  return result


$$
\mathcal{R}_{\max} =
\begin{cases}
\frac{\max(\mathrm{High}_{t+1:t+3})}{\mathrm{Close}_t} - 1 & \text{if LONG} \\[6pt]
1 - \frac{\min(\mathrm{Low}_{t+1:t+3})}{\mathrm{Close}_t} & \text{if SHORT}
\end{cases}
$$


In [5]:
def ror_max(df, decision):
  current_close = float(df.iloc[-4]["Close"])

  future_highs = (
    df.iloc[-3:]["High"].astype(float)
  )
  
  future_lows = (
    df.iloc[-3:]["Low"].astype(float)
  )

  if decision == "LONG":
    result = max(future_highs) / current_close - 1
  elif decision == "SHORT":
    result = 1 - min(future_lows) / current_close
  else:
    raise ValueError(f"Unknown decision: {decision}")

  return result


$$
\mathcal{R}_{\min} =
\begin{cases}
\frac{\min(\mathrm{Low}_{t+1:t+3})}{\mathrm{Close}_t} - 1 & \text{if LONG} \\[6pt]
1 - \frac{\max(\mathrm{High}_{t+1:t+3})}{\mathrm{Close}_t} & \text{if SHORT}
\end{cases}
$$


In [6]:
def ror_min(df, decision):
  current_close = float(df.iloc[-4]["Close"])

  future_highs = (
    df.iloc[-3:]["High"].astype(float)
  )
  
  future_lows = (
    df.iloc[-3:]["Low"].astype(float)
  )

  if decision == "LONG":
    result = min(future_lows) / current_close - 1
  elif decision == "SHORT":
    result = 1 - max(future_highs) / current_close
  else:
    raise ValueError(f"Unknown decision: {decision}")

  return result

## Baselines

### Random

Random Methods used in paper performs random selection of market trend between LONG and SHORT and risk-reward ratio $\in [1.2, 1.8]$.

In [7]:
class RandomBaseline:
  def __init__(self, seed=SEED):
    self.rng = np.random.default_rng(seed=seed)
    
  def predict(self, visible):
    return {
      "decision": self.rng.choice(["LONG", "SHORT"]),
      "risk_reward_ratio": self.rng.uniform(1.2, 1.8),
    }

### Linear Regression

Linear Regression serves as a rule-based baseline. It fits a linear model to a 40-bar window of recent closing prices and use the slope of the fitted line to classify future trend. If the slope exceeds 0, the system predicts LONG. Otherwise, it predicts SHORT.

In [8]:
class LinearRegressionBaseline:
  def __init__(self, seed=SEED):
    self.rng = np.random.default_rng(seed=seed)
    
  def predict(self, visible):
    y = visible[-40:]["Close"].astype(float).to_numpy()
    X = np.arange(40)
    
    model = LinearRegression()
    model.fit(X.reshape(-1, 1), y)
    
    slope = model.coef_[0]    
    decision = "LONG" if slope > 0 else "SHORT"
    
    return {
      "decision": decision,
      "risk_reward_ratio": self.rng.uniform(1.2, 1.8),
    }

### XGBoost

XGBoost serves as a tree-based supervised learning model. It uses technical indicators extracted via a public API TA-Lib (TA-Lib Development Team, 2025) including RSI, MACD, and SMA. This model is trained on hundreds of sliding-window sample across 50 randomly selected csv files. The trained model is tested on the rest 50 csv files with the same metrics as other methods. A majority-vote rule is applied across predictions to produce a final LONG/SHORT/HOLD decision where HOLD decision is disregarded during final average calculation.

In [9]:
def segment_files(asset, timeframe):
  candlesticks_path = REPO_ROOT / "benchmark"
  folder = candlesticks_path / asset.lower() if timeframe == "4h" else candlesticks_path / "1h" / asset.lower()
  files = sorted(folder.glob(f"{asset.upper()}_{timeframe}_*.csv"), key=lambda p: int(p.stem.split("_")[-1]))
  if not files:
    raise FileNotFoundError(f"No {asset} {timeframe} files in {folder}")
  return files

# Cached: the Random seeds and XGBoost read the same segments many times, and nothing modifies the returned frames
@functools.lru_cache(maxsize=None)
def load_segment(path):
  df = pd.read_csv(path, parse_dates=["Datetime"])
  assert len(df) == 100, f"{path.name}: expected 100 candles, got {len(df)}"
  assert df["Datetime"].is_monotonic_increasing, f"{path.name}: candles are not in time order"
  return df

In [10]:
class XGBoostBaseline:
  def __init__(self, seed=SEED):
    self.seed = seed
    self.params = {"max_depth": 3,
                   "n_estimators": 100,
                   "learning_rate": 0.1,
                  }
    self.abstain_margin = 0.0
    self.rng = np.random.default_rng(seed=seed)
    
  def _features(self, df):
    close = df["Close"].to_numpy(dtype=float)
    rsi = talib.RSI(close, timeperiod=14)
    macd, signal, histogram = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    sma = talib.SMA(close, timeperiod=20)
    return pd.DataFrame({"rsi": rsi, "macd": macd, "signal": signal, "histogram": histogram, "sma": sma}, index=df.index)
    
  def _make_samples(self, train_files):
    start = 33 # starting from row 33 so that every row have all macd metrics computed
    stop = 97 # ending at 97 so that the last candle has 3 candles after it

    X_parts = []
    Y_parts = []
    for path in train_files:
      segment = load_segment(path)
      features = self._features(segment)
      close = segment["Close"].to_numpy(dtype=float)

      close_now = close[start:stop]
      labels = dict()
      for h in range(1, 4):
        close_later = close[(start + h):(stop + h)]
        labels[f"up_{h}"] = close_later > close_now

      X_parts.append(features[start:stop])
      Y_parts.append(pd.DataFrame(labels, index=features.index[start:stop]))

    X = pd.concat(X_parts, ignore_index=True)
    Y = pd.concat(Y_parts, ignore_index=True)
    assert not X.isna().any().any(), "some features are NaN: an indicator needs more warm-up than 33 rows"
    return X, Y
    
  def fit(self, train_files):
    model_up1 = xgb.XGBClassifier(random_state=self.seed, **self.params)
    model_up2 = xgb.XGBClassifier(random_state=self.seed, **self.params)
    model_up3 = xgb.XGBClassifier(random_state=self.seed, **self.params)
    X, Y = self._make_samples(train_files)
    model_up1.fit(X, Y["up_1"])
    model_up2.fit(X, Y["up_2"])
    model_up3.fit(X, Y["up_3"])
    self.n_samples_ = len(X)
    self.models_ = [model_up1, model_up2, model_up3]
    return self
    
    
  def predict(self, visible):
    if not hasattr(self, "models_"):
      raise AttributeError("please call fit() first")
    x = self._features(visible).iloc[[-1]]
    assert not x.isna().any().any(), "some features are NaN"
    
    long_votes = 0
    short_votes = 0
    for model in self.models_:
      p = model.predict_proba(x)[0][1]
      if p > 0.5 + self.abstain_margin:
        long_votes += 1
      elif p < 0.5 - self.abstain_margin:
        short_votes += 1
    
    if long_votes > short_votes:
      decision = "LONG"
    elif short_votes > long_votes:
      decision = "SHORT"
    else:
      decision = "HOLD"
    
    return {
      "decision": decision,
      "risk_reward_ratio": self.rng.uniform(1.2, 1.8),
    }

### QuantHarness

In [11]:
QUANTHARNESS_PROVIDER = "gemini"  # "gemini" or "openai"
# Models per provider, tried in order: when one runs out of daily quota, the run cell moves on to the next
MODELS_BY_PROVIDER = {
  "gemini": ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.8-flash", "gemini-3.6-flash",
             # More free-tier models that take images and tool calls, only used once the ones above run out
             "gemini-3-flash-preview", "gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro"],
  "openai": ["gpt-4o-mini"],  # the repo default is gpt-4o
}
QUANTHARNESS_MODELS = MODELS_BY_PROVIDER[QUANTHARNESS_PROVIDER]
QUANTHARNESS_WINDOW = 45  # candles passed to the agents

class QuantHarnessBaseline:
  def __init__(self, asset, timeframe, model=QUANTHARNESS_MODELS[0], provider=QUANTHARNESS_PROVIDER, window=QUANTHARNESS_WINDOW):
    self.model = model
    self.config = DEFAULT_CONFIG.copy()     
    self.config.update({
      "agent_llm_provider": provider,
      "graph_llm_provider": provider,
      "agent_llm_model": model,
      "graph_llm_model": model,
    })     

    self.trading_graph = TradingGraph(config=self.config)
    self.asset = asset
    self.timeframe = timeframe
    self.window = window
    
    self.asset_mapping = {
      "SPX": "S&P 500",
      "BTC": "Bitcoin",
      "GC": "Gold Futures",
      "NQ": "Nasdaq Futures",
      "CL": "Crude Oil",
      "ES": "E-mini S&P 500",
      "DJI": "Dow Jones",
      "QQQ": "Invesco QQQ Trust",
      "VIX": "Volatility Index",
      "DXY": "US Dollar Index",
      "AAPL": "Apple Inc.",
      "TSLA": "Tesla Inc.",
      "DAX": "DAX Futures",
    }
    
    self.timeframe_mapping = {
      "4h": "4hour",
      "1h": "1hour",
    }
    
  def predict(self, visible):
    df_slice = visible.tail(self.window).reset_index(drop=True)
    required_columns = ["Datetime", "Open", "High", "Low", "Close"]
    if not all(col in df_slice.columns for col in required_columns):
        raise ValueError(f"Missing required columns. Available: {list(df_slice.columns)}")

    df_slice = df_slice.reset_index(drop=True)
    
    df_slice_dict = dict()
    for col in required_columns:
        if col == "Datetime":
            df_slice_dict[col] = (
                df_slice[col].dt.strftime("%Y-%m-%d %H:%M:%S").tolist()
            )
        else:
            df_slice_dict[col] = df_slice[col].tolist()
    
    p_image = static_util.generate_kline_image(df_slice_dict)
    t_image = static_util.generate_trend_image(df_slice_dict)
    
    initial_state = {
                "kline_data": df_slice_dict,
                "analysis_results": None,
                "messages": [],
                "time_frame": self.timeframe_mapping[self.timeframe],
                "stock_name": self.asset_mapping[self.asset.upper()],
                "pattern_image": p_image["pattern_image"],
                "trend_image": t_image["trend_image"],
              }
    
    try:
      final_state = self.trading_graph.graph.invoke(initial_state)
    except Exception as e:
      return {
        "decision": None,
        "risk_reward_ratio": None,
        "raw_decision": None,
        "error": f"{type(e).__name__}: {e}",
      }

    final_decision_raw = final_state.get("final_trade_decision", "")
    if isinstance(final_decision_raw, str) and final_decision_raw.lstrip().startswith("["):
      try:
        final_decision_raw = ast.literal_eval(final_decision_raw)
      except (ValueError, SyntaxError):
        pass
    if isinstance(final_decision_raw, list):
      final_decision_raw = "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in final_decision_raw)

    final_decision = {"decision": None, "risk_reward_ratio": None}
    if final_decision_raw:
      try:
        start = final_decision_raw.find("{")
        end = final_decision_raw.rfind("}") + 1
        if start != -1 and end != 0:
          json_str = final_decision_raw[start:end]
          json_str = re.sub(r",\s*}", "}", json_str)
          decision_data = json.loads(json_str)
          decision = str(decision_data.get("decision") or "").strip().upper()
          if decision not in {"LONG", "SHORT"}:
            decision = None
          try:
            risk_reward_ratio = float(str(decision_data.get("risk_reward_ratio")).split(":")[-1])
          except ValueError:
            risk_reward_ratio = None
          final_decision = {
            "decision": decision,
            "risk_reward_ratio": risk_reward_ratio,
            "forecast_horizon": decision_data.get(
                "forecast_horizon", "N/A"
            ),
            "justification": decision_data.get("justification", "N/A"),
          }
        else:
          # If no JSON found, return the raw text
          final_decision = {"decision": None, "risk_reward_ratio": None, "raw": final_decision_raw}
      except (json.JSONDecodeError, AttributeError):
        # If JSON parsing fails, return the raw text
        final_decision = {"decision": None, "risk_reward_ratio": None, "raw": final_decision_raw}

    return {
      "decision": final_decision["decision"], 
      "risk_reward_ratio": final_decision["risk_reward_ratio"], 
      "raw_decision": final_decision_raw,
      "error": None,
    }

## Comparison

In [12]:
def evaluate(model, method, asset, timeframe, files):
  res = []
  for path in files:
    candles = load_segment(path)
    visible = candles[:97]
    pred = model.predict(visible)
    res.append({"file": path.name,
                "asset": asset,
                "timeframe": timeframe,
                "method": method,
                **pred,
                "accuracy": directional_accuracy(candles, pred["decision"]),
                "ror_cc": ror_cc(candles, pred["decision"]),
                "ror_max": ror_max(candles, pred["decision"]),
                "ror_min": ror_min(candles, pred["decision"]),
                })

  return pd.DataFrame(res)

In [13]:
def split_files(files, seed=SEED, how="chronological"):
  files = list(files)
  half = len(files) // 2

  if how == "random":
    order = np.random.default_rng(seed).permutation(len(files))
    train_files = [files[i] for i in sorted(order[:half])]
    test_files = [files[i] for i in sorted(order[half:])]

  elif how == "chronological":
    start = dict()
    end = dict()
    for path in files:
      times = pd.read_csv(path, usecols=["Datetime"], parse_dates=["Datetime"])["Datetime"]
      start[path] = times.iloc[0]
      end[path] = times.iloc[-1]

    by_start = sorted(files, key=lambda p: start[p])
    train_files = by_start[:half]
    test_files = by_start[half:]
    
    test_begins = start[test_files[0]]
    train_files = [path for path in train_files if end[path] < test_begins]

  else:
    raise ValueError(f"Unknown split: {how}")

  return (train_files, test_files)

In [ ]:
results_dir = REPO_ROOT / "reproduction" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
quantharness_path = results_dir / f"quantharness_gemini-mix_w{QUANTHARNESS_WINDOW}.csv"
assets = {"4h": ["BTC", "CL", "DJI", "ES", "NQ", "QQQ", "SPX", "VIX"],
          "1h": ["BTC", "CL", "DAX", "DJI", "ES", "NQ", "QQQ", "SPX"]}
limit = None # for debug

done = set()
if quantharness_path.exists():
  saved = pd.read_csv(quantharness_path)
  done = set(saved.loc[saved["error"].isna(), "file"])

metric_functions = {"accuracy": directional_accuracy, "ror_cc": ror_cc, "ror_max": ror_max, "ror_min": ror_min}
columns = ["file", "asset", "timeframe", "method", "model", "decision", "risk_reward_ratio", "raw_decision", "error", *metric_functions]

models_left = list(QUANTHARNESS_MODELS)
break_flag = False
for timeframe, timeframe_assets in assets.items():
  for asset in timeframe_assets:
    todo = [path for path in segment_files(asset, timeframe)[:limit] if path.name not in done]
    quantharness = None
    minute_waits = 0
    while todo and models_left:
      if quantharness is None:
        quantharness = QuantHarnessBaseline(asset, timeframe, model=models_left[0])
      path = todo[0]
      candles = load_segment(path)
      pred = quantharness.predict(candles[:97])

      if pred["error"]:
        unavailable_markers = ("NOT_FOUND", "PERMISSION_DENIED", "NotFoundError", "PermissionDeniedError")
        if any(marker in pred["error"] for marker in unavailable_markers):
          print(f"{quantharness.model}: {pred['error'][:300]}\nmodel not available, switching to the next model")
          models_left.pop(0)
          quantharness = None
          minute_waits = 0
          continue
        quota_markers = ("RESOURCE_EXHAUSTED", "RateLimitError", "Max retries exceeded")
        if any(marker in pred["error"] for marker in quota_markers):
          daily = any(marker in pred["error"] for marker in ("PerDay", "per day", "insufficient_quota"))
          if not daily and minute_waits < 3:
            minute_waits += 1
            print(f"{quantharness.model}: {pred['error'][:300]}\nwaiting 60 s")
            time.sleep(60)
          else:
            print(f"{quantharness.model}: {pred['error'][:300]}\ndaily quota used up or still failing after 3 waits, switching to the next model")
            models_left.pop(0)
            quantharness = None
            minute_waits = 0
          continue
        else:
          print(pred["error"])
          break_flag = True
          break

      minute_waits = 0
      row = {"file": path.name, "asset": asset, "timeframe": timeframe, "method": "quantharness", "model": quantharness.model, **pred}
      for name, metric in metric_functions.items():
        row[name] = metric(candles, pred["decision"]) if pred["decision"] in ("LONG", "SHORT") else np.nan
      pd.DataFrame([row], columns=columns).to_csv(quantharness_path, mode="a", header=not quantharness_path.exists(), index=False)
      print(f"{path.name} ({quantharness.model}): {pred['decision'] or pred['error'] or 'no decision found in the reply'}")
      todo.pop(0)

    if break_flag or not models_left:
      break
  if not models_left:
    print("Every model in QUANTHARNESS_MODELS is out of quota or keeps failing: check the last error above, "
          "and run this cell again once the daily limits reset")
    break

# A retried segment is appended again, so keep only its latest row2323
quantharness = pd.read_csv(quantharness_path).drop_duplicates("file", keep="last")
print(f"{len(quantharness)} segments saved, {quantharness['decision'].isna().sum()} without a decision, "
      f"{quantharness['error'].notna().sum()} with an error")
print("segments per model:", quantharness["model"].value_counts().to_dict())

In [15]:
models = {"lr": LinearRegressionBaseline, "xgboost": XGBoostBaseline}
RANDOM_SEEDS = range(100)

results = []
for method, model_class in models.items():
  for timeframe, timeframe_assets in assets.items():
    for asset in timeframe_assets:
      files = segment_files(asset, timeframe)
      if method == "xgboost":
        train_files, test_files = split_files(files)
        model = model_class()
        model.fit(train_files)
        files = test_files
      else:
        model = model_class()
      results.append(evaluate(model, method, asset, timeframe, files))

all_results = pd.concat(results, ignore_index=True)
all_results.to_csv(results_dir / f"baselines_seed{SEED}.csv", index=False)

random_runs = []
for seed in RANDOM_SEEDS:
  for timeframe_index, (timeframe, timeframe_assets) in enumerate(assets.items()):
    for asset_index, asset in enumerate(timeframe_assets):
      model = RandomBaseline(seed=[seed, timeframe_index, asset_index])
      run = evaluate(model, "random", asset, timeframe, segment_files(asset, timeframe))
      random_runs.append(run.assign(seed=seed))
random_runs = pd.concat(random_runs, ignore_index=True)

quantharness_path = results_dir / f"quantharness_gemini-mix_w{QUANTHARNESS_WINDOW}.csv"
if quantharness_path.exists():
  quantharness = pd.read_csv(quantharness_path).drop_duplicates("file", keep="last")
  all_results = pd.concat([all_results, quantharness], ignore_index=True)

metrics = ["accuracy", "ror_cc", "ror_max", "ror_min"]
summary = all_results.groupby(["method", "timeframe", "asset"])[metrics].mean() * 100
# Random: average within each seed first, then across seeds; accuracy_seed_sd shows how far a single seed can be off
random_per_seed = random_runs.groupby(["seed", "timeframe", "asset"])[metrics].mean() * 100
random_per_seed.to_csv(results_dir / "random_per_seed.csv")
random_summary = random_per_seed.groupby(["timeframe", "asset"]).mean()
random_summary["accuracy_seed_sd"] = random_per_seed.groupby(["timeframe", "asset"])["accuracy"].std()
summary = pd.concat([summary, pd.concat({"random": random_summary}, names=["method"])]).sort_index()

if "model" in all_results:
  summary["models"] = (all_results.dropna(subset=["model"])
                       .groupby(["method", "timeframe", "asset"])["model"]
                       .agg(lambda m: ", ".join(f"{name} {share:.0%}" for name, share in m.value_counts(normalize=True).items())))
summary.to_csv(results_dir / f"baselines_summary_seed{SEED}.csv")

print(f"Saved {len(all_results)} rows, plus Random averaged over {len(RANDOM_SEEDS)} seeds, to {results_dir}")

Saved 3951 rows, plus Random averaged over 100 seeds, to /Users/vasilisagolikova/Git/QuantHarness/reproduction/results


In [16]:
with pd.option_context("display.max_rows", None):
  display(summary.round(3))

accuracy  ror_cc  ror_max  ror_min  \
method       timeframe asset                                       
lr           1h        BTC      49.333   0.047    0.618   -0.498   
                       CL       43.667  -0.032    0.463   -0.532   
                       DAX      49.000   0.048    0.361   -0.337   
                       DJI      52.667   0.019    0.331   -0.318   
                       ES       54.333   0.116    0.395   -0.232   
                       NQ       55.000   0.156    0.499   -0.272   
                       QQQ      53.333   0.022    0.469   -0.481   
                       SPX      54.000   0.024    0.358   -0.330   
             4h        BTC      45.000  -0.062    1.261   -1.220   
                       CL       56.333   0.117    1.334   -1.049   
                       DJI      53.000   0.164    0.842   -0.720   
                       ES       48.667   0.139    0.593   -0.506   
                       NQ       47.667   0.102    0.754   -0.698   
                       QQQ      60.667   0.295    1.207   -0.797   
                       SPX      59.000   0.357    0.991   -0.643   
                       VIX      49.667   1.263    6.462   -5.163   
quantharness 1h        BTC      46.000  -0.037    0.571   -0.545   
                       CL       49.000   0.043    0.487   -0.508   
                       DAX      54.333   0.065    0.404   -0.294   
                       DJI      56.000   0.019    0.338   -0.310   
                       ES       51.667   0.141    0.408   -0.219   
                       NQ       56.667   0.212    0.525   -0.246   
                       QQQ      51.546  -0.065    0.421   -0.536   
                       SPX      51.235   0.042    0.408   -0.339   
             4h        BTC      50.667   0.031    1.271   -1.210   
                       CL       57.333   0.097    1.308   -1.075   
                       DJI      48.000   0.085    0.799   -0.763   
                       ES       54.333   0.114    0.602   -0.497   
                       NQ       54.000   0.117    0.785   -0.667   
                       QQQ      49.000  -0.118    0.907   -1.096   
                       SPX      58.333   0.154    0.897   -0.738   
                       VIX      50.333   1.675    6.598   -5.027   
random       1h        BTC      50.367   0.006    0.564   -0.552   
                       CL       49.500   0.010    0.502   -0.494   
                       DAX      49.753   0.010    0.352   -0.347   
                       DJI      49.227  -0.006    0.320   -0.329   
                       ES       49.463   0.010    0.321   -0.306   
                       NQ       49.417  -0.022    0.371   -0.400   
                       QQQ      50.703   0.002    0.475   -0.475   
                       SPX      50.323   0.003    0.347   -0.341   
             4h        BTC      50.167   0.013    1.247   -1.234   
                       CL       50.287   0.023    1.206   -1.178   
                       DJI      49.993  -0.003    0.778   -0.784   
                       ES       50.267   0.005    0.553   -0.546   
                       NQ       49.343  -0.019    0.719   -0.734   
                       QQQ      49.890  -0.011    1.004   -1.000   
                       SPX      50.007  -0.001    0.822   -0.812   
                       VIX      48.950  -0.089    5.759   -5.866   
xgboost      1h        BTC      43.333  -0.110    0.353   -0.518   
                       CL       52.000  -0.012    0.537   -0.596   
                       DAX      48.667   0.088    0.330   -0.244   
                       DJI      48.667   0.019    0.375   -0.353   
                       ES       44.667  -0.239    0.194   -0.532   
                       NQ       55.333  -0.294    0.216   -0.625   
                       QQQ      45.333  -0.132    0.375   -0.627   
                       SPX      51.333   0.071    0.437   -0.302   
             4h        BTC      60.000   0.381    1.663   -1.033   
                  

## Comparison with the paper

4h directional accuracy against Table 2 of the paper. The figure is saved to `reproduction/presentation/figures/` for the slides.

In [17]:
paper_accuracy = pd.DataFrame({
  "random":       {"BTC": 45.0, "CL": 41.0, "DJI": 47.0, "ES": 51.0, "NQ": 43.7, "QQQ": 47.3, "SPX": 47.3, "VIX": 46.3},
  "lr":           {"BTC": 46.0, "CL": 54.3, "DJI": 52.0, "ES": 43.0, "NQ": 48.7, "QQQ": 56.0, "SPX": 59.7, "VIX": 48.7},
  "xgboost":      {"BTC": 45.3, "CL": 40.0, "DJI": 47.3, "ES": 52.0, "NQ": 47.3, "QQQ": 52.7, "SPX": 60.0, "VIX": 53.3},
  "quantharness": {"BTC": 50.7, "CL": 55.0, "DJI": 52.3, "ES": 55.0, "NQ": 55.3, "QQQ": 59.7, "SPX": 63.7, "VIX": 54.7},
})
assets_4h = list(paper_accuracy.index)

results_dir = REPO_ROOT / "reproduction" / "results"
saved_summary = pd.read_csv(results_dir / f"baselines_summary_seed{SEED}.csv", index_col=["method", "timeframe", "asset"])
my_accuracy = saved_summary.xs("4h", level="timeframe")["accuracy"].unstack("method").loc[assets_4h]

quantharness_4h = pd.read_csv(results_dir / f"quantharness_gemini-mix_w{QUANTHARNESS_WINDOW}.csv").drop_duplicates("file", keep="last").query("timeframe == '4h'")
quantharness_ci = 1.96 * 100 * quantharness_4h.groupby("asset")["accuracy"].sem().loc[assets_4h]

always_long = pd.Series({asset: 100 * np.mean([directional_accuracy(load_segment(path), "LONG") for path in segment_files(asset, "4h")])
                         for asset in assets_4h})

colors = {"mine": "#2a78d6", "paper": "#eb6834", "always_long": "#898781"}
ink = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7"}
methods = ["random", "lr", "xgboost", "quantharness"]
method_labels = {"random": "Random", "lr": "LR", "xgboost": "XGBoost", "quantharness": "QuantHarness"}
marker_style = {"markersize": 6, "markeredgecolor": "white", "markeredgewidth": 1.2, "linestyle": "none"}

def style_axis(ax, title):
  ax.set_xlim(38, 68)
  ax.set_xticks(range(40, 70, 5))
  ax.grid(axis="x", color=ink["grid"], linewidth=0.6)
  ax.set_axisbelow(True)
  for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
  ax.spines["bottom"].set_color(ink["axis"])
  ax.tick_params(axis="x", colors=ink["muted"], length=0, pad=3)
  ax.tick_params(axis="y", colors=ink["primary"], length=0)
  ax.set_xlabel("Accuracy (%)", color=ink["secondary"])
  ax.set_title(title, loc="left", color=ink["primary"], fontsize=8)

with plt.rc_context({"font.family": "Arial", "font.size": 7, "pdf.fonttype": 42}):
  fig, (ax_mean, ax_asset) = plt.subplots(1, 2, figsize=(4.6, 2.75), gridspec_kw={"wspace": 0.45})

  rows = methods + ["always_long"]
  for y, method in enumerate(rows):
    if method == "always_long":
      ax_mean.plot(always_long.mean(), y, marker="|", color=colors["always_long"], markersize=12, markeredgewidth=2, linestyle="none")
      ax_mean.text(always_long.mean() + 1.2, y, f"{always_long.mean():.1f}", va="center", color=ink["secondary"])
      continue
    mine, paper = my_accuracy[method].mean(), paper_accuracy[method].mean()
    ax_mean.plot([mine, paper], [y, y], color=ink["axis"], linewidth=1.5, zorder=1)
    ax_mean.plot(paper, y, marker="D", color=colors["paper"], zorder=3, **{**marker_style, "markersize": 5.5})
    ax_mean.plot(mine, y, marker="o", color=colors["mine"], zorder=3, **marker_style)
    if method == "quantharness":
      ax_mean.text(min(mine, paper) - 1.2, y, f"{min(mine, paper):.1f}", va="center", ha="right", color=ink["secondary"])
      ax_mean.text(max(mine, paper) + 1.2, y, f"{max(mine, paper):.1f}", va="center", color=ink["secondary"])
  ax_mean.set_yticks(range(len(rows)), [method_labels.get(m, "Always LONG") for m in rows])
  ax_mean.set_ylim(len(rows) - 0.4, -0.6)
  style_axis(ax_mean, "Average over 8 assets")

  for y, asset in enumerate(assets_4h):
    mine, paper, ci = my_accuracy.loc[asset, "quantharness"], paper_accuracy.loc[asset, "quantharness"], quantharness_ci[asset]
    ax_asset.plot(always_long[asset], y, marker="|", color=colors["always_long"], markersize=12, markeredgewidth=1.6, linestyle="none", zorder=1)
    ax_asset.plot([mine - ci, mine + ci], [y + 0.15, y + 0.15], color=colors["mine"], linewidth=1, alpha=0.6, zorder=2)
    ax_asset.plot(mine, y + 0.15, marker="o", color=colors["mine"], zorder=3, **marker_style)
    ax_asset.plot(paper, y - 0.15, marker="D", color=colors["paper"], zorder=3, **{**marker_style, "markersize": 5.5})
  ax_asset.set_yticks(range(len(assets_4h)), assets_4h)
  ax_asset.set_ylim(len(assets_4h) - 0.4, -0.6)
  style_axis(ax_asset, "QuantHarness by asset")

  legend_handles = [
    Line2D([], [], marker="o", color=colors["mine"], **marker_style),
    Line2D([], [], marker="D", color=colors["paper"], **{**marker_style, "markersize": 5.5}),
    Line2D([], [], marker="|", color=colors["always_long"], markersize=8, markeredgewidth=1.6, linestyle="none"),
  ]
  fig.legend(legend_handles, ["My reproduction", "Paper (Table 2)", "Always LONG"], loc="upper center", ncol=3, frameon=False,
             bbox_to_anchor=(0.5, 1.06), handletextpad=0.3, columnspacing=1.5, labelcolor=ink["primary"])

  figures_dir = REPO_ROOT / "reproduction" / "presentation" / "figures"
  figures_dir.mkdir(parents=True, exist_ok=True)
  fig.savefig(figures_dir / f"accuracy_vs_paper.png", dpi=300, bbox_inches="tight", facecolor="white")
  plt.show()

comparison = pd.DataFrame({"my reproduction": my_accuracy.mean(), "paper": paper_accuracy.mean()}).loc[methods]
comparison.loc["always_long"] = [always_long.mean(), np.nan]
display(comparison.round(1))
display(pd.DataFrame({"my reproduction": my_accuracy["quantharness"], "95% interval": quantharness_ci, "paper": paper_accuracy["quantharness"],
                      "always LONG": always_long}).round(1))

/var/folders/hp/dhhqxl710mg34mzwlg81f5k40000gn/T/ipykernel_95392/3290479895.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,my reproduction,paper
random,49.9,46.1
lr,52.5,51.1
xgboost,46.0,49.7
quantharness,52.8,55.8
always_long,55.8,NaN


,my reproduction,95% interval,paper,always LONG
BTC,50.7,7.7,50.7,55.7
CL,57.3,7.9,55.0,49.0
DJI,48.0,8.1,52.3,51.0
ES,54.3,8.0,55.0,58.7
NQ,54.0,8.4,55.3,58.3
QQQ,49.0,7.7,59.7,60.0
SPX,58.3,7.7,63.7,61.0
VIX,50.3,8.1,54.7,52.7


### Returns

4h returns against Table 2. VIX is left out of the comparison: its price ranges in the repo's data are about 1.8 times the paper's for every method, so its returns are on a different scale.

In [ ]:
# Table 2 of the paper: 4h returns in % per trade as (R_cc, R_max, R_min)
paper_table2 = {
  "random":       {"BTC": (-0.009, 1.220, -1.245), "CL": (-0.373, 0.970, -1.348), "DJI": (0.048, 0.755, -0.793), "ES": (-0.048, 0.538, -0.552),
                   "NQ": (-0.140, 0.646, -0.793), "QQQ": (-0.048, 0.930, -1.017), "SPX": (-0.162, 0.719, -0.862), "VIX": (0.059, 3.259, -3.157)},
  "lr":           {"BTC": (-0.066, 1.245, -1.210), "CL": (-0.114, 1.178, -1.141), "DJI": (0.149, 0.790, -0.725), "ES": (0.032, 0.553, -0.546),
                   "NQ": (0.147, 0.782, -0.670), "QQQ": (0.175, 1.113, -0.849), "SPX": (0.377, 0.960, -0.648), "VIX": (-0.140, 3.407, -3.099)},
  "xgboost":      {"BTC": (-0.050, 1.218, -1.331), "CL": (-0.056, 0.958, -1.151), "DJI": (-0.020, 0.874, -0.660), "ES": (-0.182, 0.440, -0.644),
                   "NQ": (-0.007, 0.706, -0.753), "QQQ": (0.210, 1.206, -0.973), "SPX": (0.050, 0.782, -0.712), "VIX": (0.161, 3.325, -3.110)},
  "quantharness": {"BTC": (0.089, 1.232, -1.212), "CL": (-0.008, 1.200, -1.119), "DJI": (0.163, 0.891, -0.649), "ES": (0.179, 0.613, -0.485),
                   "NQ": (0.216, 0.814, -0.639), "QQQ": (0.211, 1.052, -0.881), "SPX": (0.341, 0.965, -0.641), "VIX": (0.458, 3.872, -2.851)},
}
return_metrics = ["ror_cc", "ror_max", "ror_min"]
return_labels = {"ror_cc": "R_cc", "ror_max": "R_max", "ror_min": "R_min"}
paper_returns = {metric: pd.DataFrame({method: {asset: values[i] for asset, values in rows.items()} for method, rows in paper_table2.items()}).loc[assets_4h]
                 for i, metric in enumerate(return_metrics)}
my_returns = {metric: saved_summary.xs("4h", level="timeframe")[metric].unstack("method").loc[assets_4h, methods] for metric in return_metrics}

long_returns = {metric: pd.Series({asset: 100 * np.mean([function(load_segment(path), "LONG") for path in segment_files(asset, "4h")]) for asset in assets_4h})
                for metric, function in [("ror_cc", ror_cc), ("ror_max", ror_max), ("ror_min", ror_min)]}
for metric in return_metrics:
  my_returns[metric]["always_long"] = long_returns[metric]

comparable_assets = [asset for asset in assets_4h if asset != "VIX"]
quantharness_cc_ci = 1.96 * 100 * quantharness_4h.groupby("asset")["ror_cc"].sem().loc[comparable_assets]

def style_return_axis(ax, title, limit, step):
  ax.set_xlim(-limit, limit)
  ax.set_xticks(np.round(np.arange(-limit, limit + step / 2, step), 2))
  ax.grid(axis="x", color=ink["grid"], linewidth=0.6)
  ax.axvline(0, color=ink["axis"], linewidth=0.8, zorder=0)
  ax.set_axisbelow(True)
  for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
  ax.spines["bottom"].set_color(ink["axis"])
  ax.tick_params(axis="x", colors=ink["muted"], length=0, pad=3)
  ax.tick_params(axis="y", colors=ink["primary"], length=0)
  ax.set_xlabel("R_cc (% per trade)", color=ink["secondary"])
  ax.set_title(title, loc="left", color=ink["primary"], fontsize=8)

with plt.rc_context({"font.family": "Arial", "font.size": 7, "pdf.fonttype": 42}):
  fig, (ax_mean, ax_asset) = plt.subplots(1, 2, figsize=(4.6, 2.75), gridspec_kw={"wspace": 0.45})

  # Left: each method's R_cc averaged over the 7 comparable assets
  rows = methods + ["always_long"]
  for y, method in enumerate(rows):
    mine = my_returns["ror_cc"].loc[comparable_assets, method].mean()
    if method == "always_long":
      ax_mean.plot(mine, y, marker="|", color=colors["always_long"], markersize=12, markeredgewidth=2, linestyle="none")
      ax_mean.text(mine + 0.02, y, f"{mine:.2f}", va="center", color=ink["secondary"])
      continue
    paper = paper_returns["ror_cc"].loc[comparable_assets, method].mean()
    ax_mean.plot([mine, paper], [y, y], color=ink["axis"], linewidth=1.5, zorder=1)
    ax_mean.plot(paper, y, marker="D", color=colors["paper"], zorder=3, **{**marker_style, "markersize": 5.5})
    ax_mean.plot(mine, y, marker="o", color=colors["mine"], zorder=3, **marker_style)
    if method == "quantharness":
      ax_mean.text(min(mine, paper) - 0.02, y, f"{min(mine, paper):.2f}", va="center", ha="right", color=ink["secondary"])
      ax_mean.text(max(mine, paper) + 0.02, y, f"{max(mine, paper):.2f}", va="center", color=ink["secondary"])
  ax_mean.set_yticks(range(len(rows)), [method_labels.get(m, "Always LONG") for m in rows])
  ax_mean.set_ylim(len(rows) - 0.4, -0.6)
  style_return_axis(ax_mean, "Average over 7 assets", limit=0.3, step=0.1)

  # Right: QuantHarness R_cc per asset with its 95% interval
  for y, asset in enumerate(comparable_assets):
    mine, paper, ci = my_returns["ror_cc"].loc[asset, "quantharness"], paper_returns["ror_cc"].loc[asset, "quantharness"], quantharness_cc_ci[asset]
    ax_asset.plot(long_returns["ror_cc"][asset], y, marker="|", color=colors["always_long"], markersize=12, markeredgewidth=1.6, linestyle="none", zorder=1)
    ax_asset.plot([mine - ci, mine + ci], [y + 0.15, y + 0.15], color=colors["mine"], linewidth=1, alpha=0.6, zorder=2)
    ax_asset.plot(mine, y + 0.15, marker="o", color=colors["mine"], zorder=3, **marker_style)
    ax_asset.plot(paper, y - 0.15, marker="D", color=colors["paper"], zorder=3, **{**marker_style, "markersize": 5.5})
  ax_asset.set_yticks(range(len(comparable_assets)), comparable_assets)
  ax_asset.set_ylim(len(comparable_assets) - 0.4, -0.6)
  style_return_axis(ax_asset, "QuantHarness by asset", limit=0.6, step=0.2)

  legend_handles = [
    Line2D([], [], marker="o", color=colors["mine"], **marker_style),
    Line2D([], [], marker="D", color=colors["paper"], **{**marker_style, "markersize": 5.5}),
    Line2D([], [], marker="|", color=colors["always_long"], markersize=8, markeredgewidth=1.6, linestyle="none"),
  ]
  fig.legend(legend_handles, ["My reproduction", "Paper (Table 2)", "Always LONG"], loc="upper center", ncol=3, frameon=False,
             bbox_to_anchor=(0.5, 1.06), handletextpad=0.3, columnspacing=1.5, labelcolor=ink["primary"])

  figures_dir.mkdir(parents=True, exist_ok=True)
  fig.savefig(figures_dir / "returns_vs_paper.png", dpi=300, bbox_inches="tight", facecolor="white")
  plt.show()

# The same numbers as tables: every method's mean over the 7 comparable assets, then QuantHarness R_cc per asset
returns_comparison = pd.DataFrame({
  (return_labels[metric], source): table.loc[comparable_assets].mean()
  for metric in return_metrics for source, table in [("my reproduction", my_returns[metric]), ("paper", paper_returns[metric])]
}).loc[rows]
display(returns_comparison.round(3))
display(pd.DataFrame({"my reproduction": my_returns["ror_cc"].loc[comparable_assets, "quantharness"], "95% interval": quantharness_cc_ci,
                      "paper": paper_returns["ror_cc"].loc[comparable_assets, "quantharness"], "always LONG": long_returns["ror_cc"].loc[comparable_assets]}).round(3))

/var/folders/hp/dhhqxl710mg34mzwlg81f5k40000gn/T/ipykernel_95392/3123423388.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


R_cc                  R_max                  R_min  \
             my reproduction  paper my reproduction  paper my reproduction   
random                 0.001 -0.105           0.904  0.825          -0.898   
lr                     0.159  0.100           0.997  0.946          -0.805   
xgboost               -0.089 -0.008           0.974  0.883          -0.994   
quantharness           0.068  0.170           0.938  0.967          -0.864   
always_long            0.063    NaN           0.902    NaN          -0.900   

                     
              paper  
random       -0.944  
lr           -0.827  
xgboost      -0.889  
quantharness -0.804  
always_long     NaN

,my reproduction,95% interval,paper,always LONG
BTC,0.031,0.307,0.089,0.259
CL,0.097,0.373,-0.008,-0.312
DJI,0.085,0.240,0.163,0.096
ES,0.114,0.171,0.179,-0.006
NQ,0.117,0.217,0.216,0.033
QQQ,-0.118,0.335,0.211,0.265
SPX,0.154,0.258,0.341,0.108
